In [1]:
# ============================================================
# RGB TRAIN -> THERMAL TEST
# ALL PROTOCOLS | 5 MODELS | ONE-CELL NOTEBOOK
# Models:
#   1. resnet18
#   2. efficientnet_b0
#   3. mobilenet_v3_small
#   4. vit_b_16
#   5. convnext_tiny
# ============================================================

import os
import gc
import json
import time
import copy
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report
)

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import (
    resnet18, ResNet18_Weights,
    efficientnet_b0, EfficientNet_B0_Weights,
    mobilenet_v3_small, MobileNet_V3_Small_Weights,
    vit_b_16, ViT_B_16_Weights,
    convnext_tiny, ConvNeXt_Tiny_Weights,
)

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 300)
pd.set_option("display.max_colwidth", 300)

# ============================================================
# 1. SETTINGS
# ============================================================
SEED = 42
IMG_SIZE = 224
NUM_SAMPLE_FRAMES = 32

# Strong but practical batch sizes
# If GPU still has room, you can increase slightly later.
BATCH_SIZE_BY_MODEL = {
    "resnet18": 16,
    "efficientnet_b0": 12,
    "mobilenet_v3_small": 24,
    "vit_b_16": 8,
    "convnext_tiny": 10,
}

EPOCHS = 12
LR = 1e-4
WEIGHT_DECAY = 1e-4
PATIENCE = 4

USE_AMP = True
USE_PRETRAINED = True
USE_TORCH_COMPILE = True

# If notebook hangs at dataloader startup, set NUM_WORKERS = 0
NUM_WORKERS = 4
PIN_MEMORY = True
PERSISTENT_WORKERS = True
PREFETCH_FACTOR = 4

SAMPLE_START_FRAC = 0.10
SAMPLE_END_FRAC = 0.90

SKIP_IF_METRICS_EXISTS = True

TRAIN_MODALITY = "RGB"
TEST_MODALITY = "NIR"

MODELS_TO_RUN = [
    "resnet18",
    "efficientnet_b0",
    "mobilenet_v3_small",
    "vit_b_16",
    "convnext_tiny",
]

# All protocols
SPLIT_CONFIGS = [
    ("protocol_A", "protocol_A_train.csv", "protocol_A_val.csv", "protocol_A_test.csv"),

    ("protocol_B_4_to_4", "protocol_B_train_4_test_4.csv", None, "protocol_B_test_4_same.csv"),
    ("protocol_B_4_to_6", "protocol_B_train_4_test_6.csv", None, "protocol_B_test_6.csv"),
    ("protocol_B_4_to_8", "protocol_B_train_4_test_8.csv", None, "protocol_B_test_8.csv"),

    ("protocol_B_6_to_4", "protocol_B_train_6_test_4.csv", None, "protocol_B_test_4.csv"),
    ("protocol_B_6_to_6", "protocol_B_train_6_test_6.csv", None, "protocol_B_test_6_same.csv"),
    ("protocol_B_6_to_8", "protocol_B_train_6_test_8.csv", None, "protocol_B_test_8.csv"),

    ("protocol_B_8_to_4", "protocol_B_train_8_test_4.csv", None, "protocol_B_test_4.csv"),
    ("protocol_B_8_to_6", "protocol_B_train_8_test_6.csv", None, "protocol_B_test_6.csv"),
    ("protocol_B_8_to_8", "protocol_B_train_8_test_8.csv", None, "protocol_B_test_8_same.csv"),

    ("protocol_B_4_6_to_4", "protocol_B_train_4_6_test_4.csv", None, "protocol_B_test_4_from_4_6.csv"),
    ("protocol_B_4_6_to_6", "protocol_B_train_4_6_test_6.csv", None, "protocol_B_test_6_from_4_6.csv"),
    ("protocol_B_4_6_to_8", "protocol_B_train_4_6_test_8.csv", None, "protocol_B_test_8_from_4_6.csv"),

    ("protocol_B_4_8_to_4", "protocol_B_train_4_8_test_4.csv", None, "protocol_B_test_4_from_4_8.csv"),
    ("protocol_B_4_8_to_6", "protocol_B_train_4_8_test_6.csv", None, "protocol_B_test_6_from_4_8.csv"),
    ("protocol_B_4_8_to_8", "protocol_B_train_4_8_test_8.csv", None, "protocol_B_test_8_from_4_8.csv"),

    ("protocol_B_6_8_to_4", "protocol_B_train_6_8_test_4.csv", None, "protocol_B_test_4_from_6_8.csv"),
    ("protocol_B_6_8_to_6", "protocol_B_train_6_8_test_6.csv", None, "protocol_B_test_6_from_6_8.csv"),
    ("protocol_B_6_8_to_8", "protocol_B_train_6_8_test_8.csv", None, "protocol_B_test_8_from_6_8.csv"),

    ("protocol_B_4_6_8_to_4", "protocol_B_train_4_6_8_test_4.csv", None, "protocol_B_test_4_from_4_6_8.csv"),
    ("protocol_B_4_6_8_to_6", "protocol_B_train_4_6_8_test_6.csv", None, "protocol_B_test_6_from_4_6_8.csv"),
    ("protocol_B_4_6_8_to_8", "protocol_B_train_4_6_8_test_8.csv", None, "protocol_B_test_8_from_4_6_8.csv"),
]

CLASS_NAMES = [
    "doctor",
    "emergency",
    "fire",
    "help",
    "robbery",
    "sit_down",
    "stand_up",
]
label2id = {c: i for i, c in enumerate(CLASS_NAMES)}
id2label = {i: c for c, i in label2id.items()}
NUM_CLASSES = len(CLASS_NAMES)

# ============================================================
# 2. REPRODUCIBILITY
# ============================================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

# ============================================================
# 3. PATHS
# ============================================================
cwd = Path.cwd()
ROOT = cwd.parent if cwd.name == "notebooks" else cwd

SPLIT_DIR = ROOT / "data" / "processed" / "splits"
MANIFEST_DIR = ROOT / "data" / "processed" / "manifests"
FRAME_SUMMARY_CSV = MANIFEST_DIR / "frame_extraction_trimmed_summary.csv"

RESULTS_ROOT = ROOT / "results" / "rgb_train_thermal_test_5models"
CHECKPOINT_ROOT = ROOT / "checkpoints" / "rgb_train_thermal_test_5models"
LOG_ROOT = ROOT / "logs" / "rgb_train_thermal_test_5models"

RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)
LOG_ROOT.mkdir(parents=True, exist_ok=True)

if not FRAME_SUMMARY_CSV.exists():
    raise FileNotFoundError(f"Missing file: {FRAME_SUMMARY_CSV}")

print("ROOT:", ROOT)
print("TRAIN_MODALITY:", TRAIN_MODALITY)
print("TEST_MODALITY :", TEST_MODALITY)
print("RESULTS_ROOT:", RESULTS_ROOT)

# ============================================================
# 4. DEVICE
# ============================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
try:
    torch.set_float32_matmul_precision("high")
except Exception:
    pass

# ============================================================
# 5. LOAD FRAME SUMMARY
# ============================================================
frame_df_all = pd.read_csv(FRAME_SUMMARY_CSV)
frame_df_all = frame_df_all[frame_df_all["status"] == "success"].copy()
frame_df_all = frame_df_all[["pair_id", "modality", "output_dir", "frames_extracted"]].drop_duplicates()

# ============================================================
# 6. HELPERS
# ============================================================
def sample_indices_middle_region(n_total, n_samples, start_frac=0.10, end_frac=0.90):
    if n_total <= 0:
        return [0] * n_samples

    start_idx = int(round(start_frac * (n_total - 1)))
    end_idx = int(round(end_frac * (n_total - 1)))

    start_idx = max(0, min(start_idx, n_total - 1))
    end_idx = max(start_idx, min(end_idx, n_total - 1))

    idxs = np.linspace(start_idx, end_idx, n_samples)
    idxs = np.round(idxs).astype(int)
    return np.clip(idxs, 0, n_total - 1).tolist()

def frame_index_to_path(frame_dir, idx):
    return Path(frame_dir) / f"frame_{idx+1:06d}.jpg"

def load_split_df(csv_name):
    csv_path = SPLIT_DIR / csv_name
    if not csv_path.exists():
        raise FileNotFoundError(f"Missing split file: {csv_path}")
    df = pd.read_csv(csv_path)
    if "is_valid_pair" in df.columns:
        df = df[df["is_valid_pair"] == True].copy()
    return df.reset_index(drop=True)

def attach_labels(df_):
    out = df_.copy()
    out = out[out["gesture"].isin(CLASS_NAMES)].copy()
    out["label"] = out["gesture"].map(label2id)
    return out.reset_index(drop=True)

def attach_frame_info(df_, modality_name):
    sub = frame_df_all[frame_df_all["modality"] == modality_name].copy()
    sub = sub.rename(columns={
        "output_dir": "frame_dir",
        "frames_extracted": "num_frames_available"
    })
    sub = sub[["pair_id", "frame_dir", "num_frames_available"]].drop_duplicates()

    out = df_.merge(sub, on="pair_id", how="left")
    out = out[out["num_frames_available"].fillna(0) > 0].copy().reset_index(drop=True)
    out["num_frames_available"] = out["num_frames_available"].astype(int)
    return out

# ============================================================
# 7. TRANSFORMS
# ============================================================
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
])

# ============================================================
# 8. DATASET
# ============================================================
class FrameVotingDataset(Dataset):
    def __init__(self, df, transform, num_sample_frames=32):
        self.df = df.reset_index(drop=True).copy()
        self.transform = transform
        self.sampled_paths = []

        print(f"Building sampled frame paths for {len(self.df)} samples...")
        for _, row in self.df.iterrows():
            idxs = sample_indices_middle_region(
                int(row["num_frames_available"]),
                num_sample_frames,
                SAMPLE_START_FRAC,
                SAMPLE_END_FRAC
            )
            self.sampled_paths.append([frame_index_to_path(row["frame_dir"], i) for i in idxs])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        paths = self.sampled_paths[idx]

        frames = []
        for p in paths:
            try:
                with Image.open(p) as f:
                    img = f.convert("RGB")
            except Exception:
                img = Image.fromarray(np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8))
            frames.append(self.transform(img))

        frames = torch.stack(frames, dim=0)
        label = int(row["label"])

        meta = {
            "pair_id": row["pair_id"],
            "subject_id": row["subject_id"],
            "distance": row["distance"],
            "gesture": row["gesture"],
            "frame_dir": row["frame_dir"],
        }
        return frames, label, meta

def collate_fn(batch):
    frames = torch.stack([x[0] for x in batch], dim=0)
    labels = torch.tensor([x[1] for x in batch], dtype=torch.long)
    metas = [x[2] for x in batch]
    return frames, labels, metas

def make_loader(dataset, batch_size, shuffle):
    kwargs = dict(
        dataset=dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY and device.type == "cuda",
        collate_fn=collate_fn,
    )
    if NUM_WORKERS > 0:
        kwargs["persistent_workers"] = PERSISTENT_WORKERS
        kwargs["prefetch_factor"] = PREFETCH_FACTOR
    return DataLoader(**kwargs)

# ============================================================
# 9. MODEL FACTORY
# ============================================================
def create_model(model_name, num_classes, use_pretrained=True):
    if model_name == "resnet18":
        model = resnet18(weights=ResNet18_Weights.DEFAULT if use_pretrained else None)
        model.fc = nn.Linear(model.fc.in_features, num_classes)

    elif model_name == "efficientnet_b0":
        model = efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT if use_pretrained else None)
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)

    elif model_name == "mobilenet_v3_small":
        model = mobilenet_v3_small(weights=MobileNet_V3_Small_Weights.DEFAULT if use_pretrained else None)
        model.classifier[3] = nn.Linear(model.classifier[3].in_features, num_classes)

    elif model_name == "vit_b_16":
        model = vit_b_16(weights=ViT_B_16_Weights.DEFAULT if use_pretrained else None)
        model.heads.head = nn.Linear(model.heads.head.in_features, num_classes)

    elif model_name == "convnext_tiny":
        model = convnext_tiny(weights=ConvNeXt_Tiny_Weights.DEFAULT if use_pretrained else None)
        model.classifier[2] = nn.Linear(model.classifier[2].in_features, num_classes)

    else:
        raise ValueError(f"Unsupported model: {model_name}")

    model = model.to(device)
    model = model.to(memory_format=torch.channels_last)

    if USE_TORCH_COMPILE and hasattr(torch, "compile"):
        try:
            model = torch.compile(model, mode="reduce-overhead")
        except Exception:
            pass

    return model

def forward_video_voting(model, frames):
    B, T, C, H, W = frames.shape
    frames = frames.view(B * T, C, H, W).contiguous(memory_format=torch.channels_last)
    frame_logits = model(frames)
    frame_logits = frame_logits.view(B, T, -1)
    video_logits = frame_logits.mean(dim=1)
    return video_logits

# ============================================================
# 10. TRAIN / EVAL
# ============================================================
def run_one_epoch(model, loader, criterion, optimizer=None, scaler=None, train=True):
    model.train() if train else model.eval()

    total_loss = 0.0
    all_labels = []
    all_preds = []

    total_batches = len(loader)

    for batch_idx, (frames, labels, metas) in enumerate(loader):
        frames = frames.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        if train:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(train):
            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16,
                enabled=(USE_AMP and device.type == "cuda")
            ):
                video_logits = forward_video_voting(model, frames)
                loss = criterion(video_logits, labels)

            if train:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

        preds = video_logits.argmax(dim=1)

        total_loss += loss.item() * labels.size(0)
        all_labels.extend(labels.detach().cpu().numpy().tolist())
        all_preds.extend(preds.detach().cpu().numpy().tolist())

        if batch_idx % 10 == 0 or batch_idx == total_batches - 1:
            mode = "train" if train else "val"
            print(f"   [{mode}] batch {batch_idx+1}/{total_batches} | loss={loss.item():.4f}")

    return {
        "loss": total_loss / len(loader.dataset),
        "accuracy": accuracy_score(all_labels, all_preds),
        "macro_f1": f1_score(all_labels, all_preds, average="macro"),
        "weighted_f1": f1_score(all_labels, all_preds, average="weighted"),
    }

@torch.no_grad()
def predict_with_metadata(model, loader):
    model.eval()

    all_labels = []
    all_preds = []
    rows = []

    start_time = time.time()
    total_samples = 0

    for frames, labels, metas in loader:
        frames = frames.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(USE_AMP and device.type == "cuda")
        ):
            video_logits = forward_video_voting(model, frames)
            probs = torch.softmax(video_logits, dim=1)
            preds = probs.argmax(dim=1)

        probs_np = probs.detach().cpu().numpy()
        preds_np = preds.detach().cpu().numpy()
        labels_np = labels.detach().cpu().numpy()

        total_samples += len(labels_np)
        all_labels.extend(labels_np.tolist())
        all_preds.extend(preds_np.tolist())

        for i, meta in enumerate(metas):
            row = {
                "pair_id": meta["pair_id"],
                "subject_id": meta["subject_id"],
                "distance": meta["distance"],
                "gesture_true": id2label[int(labels_np[i])],
                "gesture_pred": id2label[int(preds_np[i])],
                "frame_dir": meta["frame_dir"],
            }
            for c_idx, c_name in id2label.items():
                row[f"prob_{c_name}"] = float(probs_np[i][c_idx])
            rows.append(row)

    elapsed = time.time() - start_time
    videos_per_sec = total_samples / elapsed if elapsed > 0 else None
    return all_labels, all_preds, pd.DataFrame(rows), elapsed, videos_per_sec

# ============================================================
# 11. RUN ONE EXPERIMENT
# ============================================================
def run_experiment(protocol_name, train_csv_name, val_csv_name, test_csv_name, model_name):
    print("\n" + "=" * 100)
    print(f"RUNNING | protocol={protocol_name} | train={TRAIN_MODALITY} | test={TEST_MODALITY} | model={model_name}")
    print("=" * 100)

    train_df = load_split_df(train_csv_name)
    test_df = load_split_df(test_csv_name)

    if val_csv_name is None:
        train_df = train_df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
        n_val = max(1, int(0.10 * len(train_df)))
        val_df = train_df.iloc[:n_val].copy().reset_index(drop=True)
        train_df = train_df.iloc[n_val:].copy().reset_index(drop=True)
    else:
        val_df = load_split_df(val_csv_name)

    train_df = attach_labels(train_df)
    val_df = attach_labels(val_df)
    test_df = attach_labels(test_df)

    # RGB for training/validation, NIR for testing
    train_df = attach_frame_info(train_df, TRAIN_MODALITY)
    val_df = attach_frame_info(val_df, TRAIN_MODALITY)
    test_df = attach_frame_info(test_df, TEST_MODALITY)

    print(f"train={len(train_df)}, val={len(val_df)}, test={len(test_df)}")

    batch_size = BATCH_SIZE_BY_MODEL[model_name]

    train_loader = make_loader(
        FrameVotingDataset(train_df, train_transform, NUM_SAMPLE_FRAMES),
        batch_size=batch_size,
        shuffle=True
    )
    val_loader = make_loader(
        FrameVotingDataset(val_df, eval_transform, NUM_SAMPLE_FRAMES),
        batch_size=batch_size,
        shuffle=False
    )
    test_loader = make_loader(
        FrameVotingDataset(test_df, eval_transform, NUM_SAMPLE_FRAMES),
        batch_size=batch_size,
        shuffle=False
    )

    run_name = f"{protocol_name}__rgb_train__thermal_test__{model_name}__f{NUM_SAMPLE_FRAMES}__img{IMG_SIZE}"
    results_dir = RESULTS_ROOT / run_name
    checkpoint_dir = CHECKPOINT_ROOT / run_name
    log_dir = LOG_ROOT / run_name

    results_dir.mkdir(parents=True, exist_ok=True)
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    log_dir.mkdir(parents=True, exist_ok=True)

    metrics_json = results_dir / "metrics.json"
    if SKIP_IF_METRICS_EXISTS and metrics_json.exists():
        print("Skipping, already finished:", run_name)
        with open(metrics_json, "r") as f:
            return json.load(f)

    model = create_model(model_name, NUM_CLASSES, use_pretrained=USE_PRETRAINED)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scaler = torch.amp.GradScaler("cuda", enabled=(USE_AMP and device.type == "cuda"))

    best_val_f1 = -1.0
    best_epoch = -1
    epochs_without_improvement = 0
    history = []

    best_ckpt_path = checkpoint_dir / "best_model.pt"
    best_state_dict = None

    train_start = time.time()

    for epoch in range(1, EPOCHS + 1):
        print(f"\nEpoch {epoch}/{EPOCHS}")

        train_metrics = run_one_epoch(model, train_loader, criterion, optimizer, scaler, train=True)
        val_metrics = run_one_epoch(model, val_loader, criterion, optimizer=None, scaler=scaler, train=False)

        history.append({
            "epoch": epoch,
            "train_loss": train_metrics["loss"],
            "train_acc": train_metrics["accuracy"],
            "train_macro_f1": train_metrics["macro_f1"],
            "train_weighted_f1": train_metrics["weighted_f1"],
            "val_loss": val_metrics["loss"],
            "val_acc": val_metrics["accuracy"],
            "val_macro_f1": val_metrics["macro_f1"],
            "val_weighted_f1": val_metrics["weighted_f1"],
        })

        print(
            f"Epoch {epoch:02d} | "
            f"train_macro_f1={train_metrics['macro_f1']:.4f} | "
            f"val_macro_f1={val_metrics['macro_f1']:.4f}"
        )

        if val_metrics["macro_f1"] > best_val_f1:
            best_val_f1 = val_metrics["macro_f1"]
            best_epoch = epoch
            epochs_without_improvement = 0
            best_state_dict = copy.deepcopy(model.state_dict())
            torch.save({
                "epoch": epoch,
                "model_state_dict": best_state_dict,
                "model_name": model_name,
                "train_modality": TRAIN_MODALITY,
                "test_modality": TEST_MODALITY,
                "protocol_name": protocol_name,
                "label2id": label2id,
                "id2label": id2label,
            }, best_ckpt_path)
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= PATIENCE:
            print("Early stopping triggered.")
            break

    train_elapsed = time.time() - train_start
    pd.DataFrame(history).to_csv(log_dir / "training_log.csv", index=False)

    if best_state_dict is None:
        raise RuntimeError("No best model was saved.")

    model.load_state_dict(best_state_dict)

    test_labels, test_preds, pred_df, infer_elapsed, videos_per_sec = predict_with_metadata(model, test_loader)

    test_acc = accuracy_score(test_labels, test_preds)
    test_macro_f1 = f1_score(test_labels, test_preds, average="macro")
    test_weighted_f1 = f1_score(test_labels, test_preds, average="weighted")

    cm = confusion_matrix(test_labels, test_preds)
    report_dict = classification_report(
        test_labels,
        test_preds,
        target_names=[id2label[i] for i in range(NUM_CLASSES)],
        output_dict=True,
        zero_division=0
    )
    report_df = pd.DataFrame(report_dict).transpose()

    pred_csv = results_dir / "test_predictions.csv"
    report_csv = results_dir / "classification_report.csv"
    cm_png = results_dir / "confusion_matrix.png"

    pred_df.to_csv(pred_csv, index=False)
    report_df.to_csv(report_csv)

    plt.figure(figsize=(8, 6))
    plt.imshow(cm, interpolation="nearest")
    plt.title(f"{protocol_name} | RGB train | Thermal test | {model_name}")
    plt.colorbar()
    ticks = np.arange(NUM_CLASSES)
    plt.xticks(ticks, [id2label[i] for i in range(NUM_CLASSES)], rotation=45, ha="right")
    plt.yticks(ticks, [id2label[i] for i in range(NUM_CLASSES)])
    plt.xlabel("Predicted")
    plt.ylabel("True")
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, str(cm[i, j]), ha="center", va="center")
    plt.tight_layout()
    plt.savefig(cm_png, dpi=200, bbox_inches="tight")
    plt.close()

    per_class_f1 = {}
    for cls in CLASS_NAMES:
        if cls in report_df.index:
            per_class_f1[cls] = float(report_df.loc[cls, "f1-score"])

    metrics = {
        "protocol_name": protocol_name,
        "train_modality": TRAIN_MODALITY,
        "test_modality": TEST_MODALITY,
        "model_name": model_name,
        "num_sample_frames": NUM_SAMPLE_FRAMES,
        "batch_size": batch_size,
        "best_epoch": int(best_epoch),
        "best_val_macro_f1": float(best_val_f1),
        "test_accuracy": float(test_acc),
        "test_macro_f1": float(test_macro_f1),
        "test_weighted_f1": float(test_weighted_f1),
        "per_class_f1": per_class_f1,
        "train_time_minutes": float(train_elapsed / 60.0),
        "inference_time_seconds": float(infer_elapsed),
        "videos_per_second": float(videos_per_sec) if videos_per_sec is not None else None,
        "train_rows": int(len(train_df)),
        "val_rows": int(len(val_df)),
        "test_rows": int(len(test_df)),
        "predictions_csv": str(pred_csv),
        "classification_report_csv": str(report_csv),
        "confusion_matrix_png": str(cm_png),
        "checkpoint_best": str(best_ckpt_path),
    }

    with open(metrics_json, "w") as f:
        json.dump(metrics, f, indent=4)

    print(json.dumps(metrics, indent=2))

    del model, train_loader, val_loader, test_loader, best_state_dict
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return metrics

# ============================================================
# 12. RUN ALL EXPERIMENTS
# ============================================================
summary_csv = RESULTS_ROOT / "rgb_train_thermal_test_5models_summary.csv"

all_metrics = []
if summary_csv.exists():
    try:
        prev = pd.read_csv(summary_csv)
        all_metrics = prev.to_dict(orient="records")
    except Exception:
        all_metrics = []

done_keys = set()
for m in all_metrics:
    done_keys.add((m["protocol_name"], m["model_name"]))

for protocol_name, train_csv_name, val_csv_name, test_csv_name in SPLIT_CONFIGS:
    for model_name in MODELS_TO_RUN:
        key = (protocol_name, model_name)
        if SKIP_IF_METRICS_EXISTS and key in done_keys:
            print("Already in summary, skipping:", key)
            continue

        metrics = run_experiment(
            protocol_name=protocol_name,
            train_csv_name=train_csv_name,
            val_csv_name=val_csv_name,
            test_csv_name=test_csv_name,
            model_name=model_name
        )
        all_metrics.append(metrics)
        done_keys.add(key)
        pd.DataFrame(all_metrics).to_csv(summary_csv, index=False)

final_df = pd.DataFrame(all_metrics)
final_df.to_csv(summary_csv, index=False)

print("\nDONE.")
print("Saved summary to:", summary_csv)
display(
    final_df[[
        "protocol_name",
        "model_name",
        "best_val_macro_f1",
        "test_accuracy",
        "test_macro_f1",
        "test_weighted_f1",
        "train_time_minutes",
        "videos_per_second"
    ]].sort_values(["protocol_name", "test_macro_f1"], ascending=[True, False])
)

ROOT: /data/Sajjan_Singh/gesture_recognition
TRAIN_MODALITY: RGB
TEST_MODALITY : NIR
RESULTS_ROOT: /data/Sajjan_Singh/gesture_recognition/results/rgb_train_thermal_test_5models
Using device: cuda
GPU name: NVIDIA H100 NVL

RUNNING | protocol=protocol_A | train=RGB | test=NIR | model=resnet18
train=525, val=77, test=147
Building sampled frame paths for 525 samples...
Building sampled frame paths for 77 samples...
Building sampled frame paths for 147 samples...

Epoch 1/12
   [train] batch 1/33 | loss=2.2530
   [train] batch 11/33 | loss=1.6949
   [train] batch 21/33 | loss=1.6292
   [train] batch 31/33 | loss=1.8552
   [train] batch 33/33 | loss=1.4911
   [val] batch 1/5 | loss=0.8420
   [val] batch 5/5 | loss=1.1937
Epoch 01 | train_macro_f1=0.2514 | val_macro_f1=0.5999

Epoch 2/12
   [train] batch 1/33 | loss=0.9511
   [train] batch 11/33 | loss=0.9119
   [train] batch 21/33 | loss=0.8399
   [train] batch 31/33 | loss=0.8200
   [train] batch 33/33 | loss=0.7677
   [val] batch 1/5 | lo

,protocol_name,model_name,best_val_macro_f1,test_accuracy,test_macro_f1,test_weighted_f1,train_time_minutes,videos_per_second
4,protocol_A,convnext_tiny,0.895889,0.421769,0.350680,0.350680,39.857117,11.084656
2,protocol_A,mobilenet_v3_small,0.569607,0.244898,0.175473,0.175473,75.003222,8.862384
1,protocol_A,efficientnet_b0,0.822656,0.217687,0.150062,0.150062,72.370999,10.603398
0,protocol_A,resnet18,0.870092,0.190476,0.117037,0.117037,39.486911,3.897324
3,protocol_A,vit_b_16,0.035714,0.142857,0.035714,0.035714,28.648462,10.928736
...,...,...,...,...,...,...,...,...
49,protocol_B_8_to_8,convnext_tiny,0.871429,0.364662,0.308082,0.308082,43.064069,10.476125
47,protocol_B_8_to_8,mobilenet_v3_small,0.162698,0.142857,0.091388,0.091388,39.882645,9.088298
46,protocol_B_8_to_8,efficientnet_b0,0.559524,0.176692,0.088268,0.088268,51.798239,9.128603
45,protocol_B_8_to_8,resnet18,0.584014,0.154135,0.053414,0.053414,53.972139,9.586131
